# Aura — training the wellness coach on Gemma 3n

This notebook drives the `aura` package rather than redefining the pipeline inline.
Everything here is importable, tested code — the notebook is the demo, not the source
of truth.

**Runtime:** GPU required for training (a Colab T4 is enough for the E2B model).
The inference and evaluation sections run on CPU.

| Section | Needs a GPU |
|---|---|
| 1. Setup | no |
| 2. Build the preference dataset | no |
| 3. Inspect what the model will learn | no |
| 4. Fine-tune with DPO | **yes** |
| 5. Evaluate behaviour | no (CPU is slow but works) |
| 6. Serve it | no |

## 1. Setup

In [ ]:
# In Colab, clone the repo. Locally, skip this cell.
import os, sys, subprocess

IN_COLAB = 'COLAB_' in ''.join(os.environ.keys())
if IN_COLAB and not os.path.exists('Gemma-3n-Hackathon'):
    !git clone https://github.com/arnavdsp/Gemma-3n-Hackathon.git
    %cd Gemma-3n-Hackathon

# `[train]` pulls unsloth, trl, peft and datasets. Omit it for a CPU-only run.
EXTRAS = '[train]' if IN_COLAB else ''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'.{EXTRAS}'], check=False)

In [ ]:
from aura.logging import configure_logging
configure_logging('INFO')

import aura
print('aura', aura.__version__)

## 2. Build the preference dataset

Two sources, one preference: **reflective and curious** over **dismissive and directive**.

- `jkhedri/psychology-dataset` — has its own preferred (`response_j`) and dispreferred
  (`response_k`) answers. We append a coaching invitation to the preferred one so the
  model also learns to hand the turn back rather than close it down.
- `drublackberry/hbr-coaching-real-leaders` — real coach turns become `chosen`; a generic
  directive reply becomes `rejected`. The rejection deliberately carries **no content**
  from the prompt, so the signal is the stance rather than any leaked wording.

`build_preference_dataset` deduplicates, length-filters, and reports why rows were dropped.

In [ ]:
from aura.training.data import build_preference_dataset

train_ds, eval_ds = build_preference_dataset(psychology_limit=3000, eval_fraction=0.05)
print(f'train: {len(train_ds)} pairs   eval: {len(eval_ds) if eval_ds else 0} pairs')

## 3. Inspect what the model will learn

Always read the data before training on it.

In [ ]:
import textwrap

def show(row):
    for field in ('prompt', 'chosen', 'rejected'):
        print(f'--- {field.upper()} ---')
        print(textwrap.fill(row[field][:600], 92))
        print()

show(train_ds[0])

In [ ]:
# The rejected side should never quote the prompt back. Verify that holds.
import random

leaks = 0
for row in random.Random(0).sample(list(train_ds), min(200, len(train_ds))):
    prompt_words = {w for w in row['prompt'].lower().split() if len(w) > 6}
    rejected_words = set(row['rejected'].lower().split())
    if prompt_words & rejected_words:
        leaks += 1
print(f'{leaks} of 200 sampled rejections share a distinctive word with their prompt')

## 4. Fine-tune with DPO

DPO because the target behaviour is a *preference between two plausible replies*,
not a single correct string.

Defaults are tuned for a 16 GB GPU. Raise `max_steps` for a real run — 200 steps is
a smoke test, not a finished model.

In [ ]:
from pathlib import Path
from aura.training.train import TrainingConfig, run

config = TrainingConfig(
    model_id='unsloth/gemma-3n-E2B-it',
    output_dir=Path('artifacts/wellness-coach'),
    strategy='dpo',
    fallback_to_sft=True,   # drop to SFT if DPO runs out of memory
    max_steps=200,
    learning_rate=5e-6,
    lora_rank=16,
    beta=0.1,
)
config

In [ ]:
# Uncomment to train. This takes roughly 30-45 minutes on a T4 at 200 steps.
# adapter_path = run(config)
# print('adapter saved to', adapter_path)

## 5. Evaluate behaviour, not just loss

A falling loss curve doesn't tell you whether the coach got better. These criteria do:
does it ask a question, avoid directives, reflect the speaker, hedge its reads, stay
brief, and avoid diagnosing? Plus a pass/fail safety check.

The criteria are lexical proxies — good for catching regressions between checkpoints,
not ground truth. Only safety is treated as pass/fail.

In [ ]:
import asyncio

from aura.config import Settings
from aura.engine.base import GenerationRequest
from aura.engine.registry import build_engine
from aura.prompts import build_system_prompt
from aura.safety import crisis_message, screen
from aura.schemas import AffectSignal
from aura.training.evaluate import evaluate

# Swap engine='gemma' and adapter_path=... to score a trained checkpoint.
settings = Settings(engine='echo')
engine = build_engine(settings)
loop = asyncio.new_event_loop()
loop.run_until_complete(engine.warmup())

SYSTEM = build_system_prompt(None, AffectSignal())

def generate(prompt: str) -> str:
    assessment = screen(prompt, region=settings.crisis_region)
    if assessment.should_short_circuit:
        return crisis_message(assessment)
    return loop.run_until_complete(
        engine.generate(GenerationRequest(system_prompt=SYSTEM, user_text=prompt))
    )

report = evaluate(generate)
print(report.render())

In [ ]:
import textwrap

# Read the actual replies. Numbers hide a lot.
for score in report.scores[:3]:
    print('USER :', score.prompt)
    print('AURA :', textwrap.fill(score.response, 92))
    print(f'score: {score.total:.0%}\n')

### The safety screen runs before generation

This is the property that matters most: on a crisis match the model is never invoked,
so no prompt can route around it.

In [ ]:
from aura.safety import screen

examples = [
    "I've been feeling anxious about work",
    "I feel completely hopeless",
    "I don't want to be here anymore",
    "I wanted to die years ago, but I'm past that now",  # softened, not a crisis
    "I'm going to kill this presentation tomorrow",
]
for text in examples:
    a = screen(text, region='IN')
    print(f'{a.risk.value:<9} short_circuit={str(a.should_short_circuit):<5} | {text}')

## 6. Serve it

```bash
AURA_ENGINE=gemma \
AURA_ADAPTER_PATH=artifacts/wellness-coach/adapter \
aura serve
```

Then open <http://localhost:8000> for the full multimodal interface — voice input with
a live waveform, image sharing, spoken replies, and the session insights panel.

In Colab, expose it with a tunnel:

```python
!pip install -q pyngrok
from pyngrok import ngrok
print(ngrok.connect(8000))
```

In [ ]:
# Drive the coach directly, without the HTTP layer.
from aura.coach import Coach
from aura.schemas import ChatRequest
from aura.session import SessionStore

store = SessionStore(settings)
coach = Coach(settings, engine, store)

session_id = None
for message in [
    'Work has been relentless and I have not slept properly in a week.',
    'My partner says I am never present any more. I feel alone in it.',
]:
    prepared, reply, _ = loop.run_until_complete(coach.respond(
        ChatRequest(session_id=session_id, message=message)
    ))
    session_id = prepared.memory.session_id
    print('USER :', message)
    print('AURA :', textwrap.fill(reply.text, 92))
    print(f'read as: {prepared.affect.label} ({prepared.affect.confidence:.0%} confident)\n')

memory = loop.run_until_complete(store.get(session_id))
print('topic graph :', memory.graph.as_dict()['topics'])
print('mood trend  :', [round(v, 2) for v in memory.mood_trend])
print('context note:', memory.context_note())